# Background / Anatomical region / Organs / Other-tissue Debug

This notebook loads one case, runs background and anatomical-region detection plus Total Segmentator organ masks, builds the combined mask, and visualizes each layer to debug and fix the pipeline.

**Target semantics:**
1. **Background** (0) – outside patient
2. **Anatomical region** (1) – patient outline (head or body)
3. **Organs** (3+) – Total Segmentator labels
4. **Other-tissue** (2) – inside anatomical region but not an organ

## Setup

Add project root to path and import required modules.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Project root (parent of research_notebooks when running from notebook dir)
project_root = Path.cwd().parent if Path.cwd().name == "research_notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from image_processor.core.mask_generator import MaskGenerator
from image_processor.utils.organ_dictionary import OrganDictionary
from image_processor.utils.image_processing import ImageProcessor
from image_processor.io.nifti_handler import NIfTIHandler

# Paths (set MAIN_PATH to your data root, e.g. containing TotalSegmentatorRetrain)
MAIN_PATH = os.environ.get("MAIN_PATH", str(project_root))
print(f"Project root: {project_root}")
print(f"MAIN_PATH: {MAIN_PATH}")

## Load one case

Use an already-processed case under `{MAIN_PATH}/TotalSegmentatorRetrain/{CASE_ID}`. The case folder should contain the CT NIfTI, Total Segmentator output subfolder(s), and optionally the tumor mask. Define `case_folder`, `nifti_ct_path`, `ts_output_path`, and `slices_to_use`. If no case is available, run `process_case` once or place a case folder manually.

In [ ]:
# Configuration: set CASE_ID to an already-processed case folder name (e.g. RADCURE-0005)
CASE_ID = "RADCURE-0005"
case_folder = os.path.join(MAIN_PATH, "TotalSegmentatorRetrain", CASE_ID)
nifti_ct_path = os.path.join(case_folder, f"{CASE_ID}.nii.gz")

# Total Segmentator output is under case_folder/total_segmentator_output/ with one subdir per task
ts_output_path = os.path.join(case_folder, "total_segmentator_output")

# Slices to use: typically tumor region expanded. If we have a saved mask we could load it; here we use a range for the full volume or a placeholder.
nii = NIfTIHandler.load_nii_image(nifti_ct_path)
n_slices = nii.shape[2]
# Placeholder: use middle third of volume; replace with non_zero_tumor_mask_expanded from process_case in production
slices_to_use = list(range(n_slices // 3, 2 * n_slices // 3))

print(f"Case folder: {case_folder}")
print(f"CT NIfTI: {nifti_ct_path}, exists: {os.path.exists(nifti_ct_path)}")
print(f"TS output path: {ts_output_path}, exists: {os.path.exists(ts_output_path)}")
print(f"Slices to use: {len(slices_to_use)} slices (indices {slices_to_use[0]}..{slices_to_use[-1]})")

## Generate background / anatomical_region per slice

For each slice, load the 2D CT and call `ImageProcessor.head_mask_from_array(img)` to get a boolean background mask (True = background). Anatomical region mask is its inverse. Visualize a few slices.

In [ ]:
# Build list of background masks (True = background) and anatomical_region (False = anatomical region in head_mask_from_array)
background_masks = []
anatomical_region_masks = []
for slice_ind in slices_to_use:
    img = nii[:, :, slice_ind]
    bg_mask = ImageProcessor.head_mask_from_array(img)
    background_masks.append(bg_mask)
    anatomical_region_masks.append(~bg_mask)

# Visualize a few slices (first, middle, last)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, idx in zip(axes, [0, len(slices_to_use) // 2, len(slices_to_use) - 1]):
    si = slices_to_use[idx]
    img = nii[:, :, si]
    ax.imshow(img, cmap="gray")
    # Overlay anatomical region in semi-transparent color
    ar = anatomical_region_masks[idx].astype(float)
    ar[ar == 0] = np.nan
    ax.imshow(ar, cmap="Reds", alpha=0.4)
    ax.set_title(f"Slice {si}: CT + anatomical region")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Inspect Total Segmentator outputs

List organ NIfTI paths under `ts_output_path` (same logic as `get_individual_segmentator_paths`). Optionally visualize one organ on the same slices.

In [ ]:
# List all organ mask paths (same logic as MaskGenerator.get_individual_segmentator_paths)
individual_paths = []
if os.path.exists(ts_output_path):
    for sec_dir in os.listdir(ts_output_path):
        sec_path = os.path.join(ts_output_path, sec_dir)
        if os.path.isdir(sec_path):
            for f in os.listdir(sec_path):
                if f.endswith(".nii.gz"):
                    individual_paths.append(os.path.join(sec_path, f))
print(f"Found {len(individual_paths)} organ mask files under {ts_output_path}")
if individual_paths:
    print("First 5:", individual_paths[:5])

## Build combined mask (current logic)

Call `MaskGenerator.generate_background_array` and `generate_combined_mask`, then stack into a 3D label volume and visualize (background / anatomical_region / organs / rest).

In [ ]:
# Organ dictionary path (optional; if None, uses default background/other-tissue)
organ_dictionary_path = os.environ.get("ORGAN_DICTIONARY_PATH") or os.path.join(MAIN_PATH, "radcure_dictionary.json")
organ_dict = OrganDictionary(organ_dictionary_path)
mask_gen = MaskGenerator(organ_dict)

# Generate background array and combined mask (requires ts_output_path to exist with organ NIfTIs)
background_array_int = mask_gen.generate_background_array(slices_to_use, nifti_ct_path)
if os.path.exists(ts_output_path) and individual_paths:
    combined_mask_list, _ = mask_gen.generate_combined_mask(
        slices_to_use, background_array_int, ts_output_path
    )
    combined_3d = np.stack(combined_mask_list, axis=-1)
    unique_labels = np.unique(combined_3d)
    print(f"Unique labels in combined mask: {unique_labels}")
    # Visualize one slice with label overlay
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    mid = len(slices_to_use) // 2
    ax.imshow(nii[:, :, slices_to_use[mid]], cmap="gray")
    labels_slice = combined_3d[:, :, mid]
    labels_slice = np.ma.masked_where(labels_slice == 0, labels_slice)
    ax.imshow(labels_slice, cmap="nipy_spectral", alpha=0.6)
    ax.set_title("Combined mask (current logic) overlay")
    ax.axis("off")
    plt.show()
else:
    print("Skipping generate_combined_mask (no TS output). Combined mask = background array only.")
    combined_3d = np.stack(background_array_int, axis=-1)

## Define desired label semantics

**Target:** 0 = background, 1 = anatomical_region, 2 = other-tissue, 3+ = organs (and later tumor).

**Current (before fix):** `generate_background_array` returns 1 = background, 0 = head; combined mask then overwrites with organ indices, so 0/1 and organ indices are mixed and "other-tissue" is never assigned. Compare current mask values to the target and note mismatches below.

In [ ]:
# Compare current semantics vs target
# Target: 0=background, 1=anatomical_region, 2=other-tissue, 3+=organs
if "combined_3d" in dir() and combined_3d is not None:
    u = np.unique(combined_3d)
    print("Current unique labels:", u)
    print("Target: 0=background, 1=anatomical_region, 2=other-tissue, 3+=organs")
    print("After code fix, generate_background_array will output 0=bg, 1=anatomical_region; generate_combined_mask will assign 2 to remaining anatomical_region pixels.")

## Experimentation / fix

Try different `head_mask_from_array` parameters (e.g. `keep_top_ratio`, `do_split`, `head_top_ratio`) to improve detection across head and shoulder/body regions. Once behavior is correct here, the same logic is applied in `mask_generator.py` and `image_processing.py`.

In [ ]:
# Example: recompute background/anatomical_region with different parameters for one slice
# ImageProcessor.head_mask_from_array(img, edge_pct=90, roi_radius=0.42, close_radius=7, min_area=2000,
#     keep_top_ratio=0.70, sigma=1.0, do_split=True, head_top_ratio=0.5)
# Adjust keep_top_ratio (e.g. 1.0 for body) or do_split=False to test.
slice_test_idx = len(slices_to_use) // 2
img_test = nii[:, :, slices_to_use[slice_test_idx]]
bg_test = ImageProcessor.head_mask_from_array(img_test, keep_top_ratio=0.8)
plt.figure(figsize=(6, 6))
plt.imshow(img_test, cmap="gray")
plt.imshow(np.ma.masked_where(~bg_test, np.ones_like(bg_test)), cmap="Blues", alpha=0.3)
plt.title("Background (blue) vs anatomical region (transparent) - adjustable params")
plt.axis("off")
plt.show()